# 🖼️ Clasificación de Imágenes CIFAR-10 con CNN + GridSearch + MLflow

---

## ¿De qué trata este cuaderno?

Aprenderás a entrenar una **Red Neuronal Convolucional (CNN)** sobre el dataset CIFAR-10,
buscar los mejores hiperparámetros con **GridSearch**, y registrar todo el proceso
automáticamente en **MLflow** para comparar experimentos.

## Flujo del experimento

```
CIFAR-10 (60,000 imágenes 32×32)
         │
         ▼
   Preprocesamiento
   (normalización + one-hot)
         │
         ▼
   GridSearch manual
   ┌─────┴──────┐
   │  combinación│  ← learning_rate × batch_size × filtros
   │  de params  │
   └─────┬──────┘
         │  Por cada combinación:
         ▼
   mlflow.start_run()
         │
         ├── log_params()     ← hiperparámetros
         ├── CNN.fit()        ← entrenamiento con EarlyStopping
         ├── log_metric()     ← accuracy/loss por época
         └── log_model()      ← mejor modelo guardado
         │
         ▼
   MLflow UI → comparar runs
```

## Entorno

| Componente | Detalle |
|---|---|
| Servidor | JupyterHub `https://unab-n8n.duckdns.org:8000/user/adiaz/` |
| MLflow | `https://unab-n8n.duckdns.org/mlflow` |
| Python | 3.12 |
| Framework DL | TensorFlow / Keras |

> ⚠️ **Nota sobre TensorBoard vs MLflow:** Para este ejercicio usaremos solo MLflow.
> MLflow registra métricas por época, parámetros, artefactos y permite comparar
> experimentos — todo lo que necesitamos. TensorBoard añadiría histogramas de pesos
> pero complejidad extra que no aporta al aprendizaje de GridSearch.

---
# 🔍 Sección 1: Diagnóstico del entorno del servidor

In [ ]:
import sys
print(f"🐍 Python: {sys.version}")
print(f"📂 Ejecutable: {sys.executable}\n")

packages = {
    "tensorflow": "Framework de deep learning",
    "mlflow":     "Tracking de experimentos",
    "numpy":      "Operaciones numéricas",
    "matplotlib": "Visualización",
    "sklearn":    "Métricas adicionales",
}

for pkg, desc in packages.items():
    try:
        mod = __import__(pkg)
        v = getattr(mod, '__version__', 'ok')
        print(f"  ✅ {pkg:<15} {v:<10}  {desc}")
    except ImportError:
        print(f"  ❌ {pkg:<15} NO instalado  — {desc}")

In [ ]:
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU disponible: {gpus}")
    print("   El entrenamiento será significativamente más rápido.")
else:
    print("⚠️  Sin GPU — se usará CPU.")
    print("   El GridSearch completo puede tardar 15-30 min.")
    print("   Usaremos un subconjunto reducido del dataset para la clase.")

---
# 📦 Sección 2: Instalación y configuración

In [ ]:
import subprocess, sys

def instalar(paquete):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", paquete, "--quiet"],
        capture_output=True, text=True
    )
    estado = "✅" if r.returncode == 0 else "❌"
    print(f"{estado} {paquete}")
    if r.returncode != 0:
        print(r.stderr[:200])

# Instala solo lo que falte según el diagnóstico
# instalar("tensorflow")   # Descomentar si no está instalado
print("Dependencias verificadas — si falta alguna, descomenta la línea correspondiente.")

In [ ]:
import os
import mlflow
import urllib3
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Silenciar advertencias SSL (entorno de desarrollo)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'

# Reducir verbosidad de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Conexión a MLflow
MLFLOW_URI = "https://unab-n8n.duckdns.org/mlflow"
mlflow.set_tracking_uri(MLFLOW_URI)

try:
    mlflow.search_experiments()
    print(f"✅ MLflow conectado: {MLFLOW_URI}")
except Exception as e:
    print(f"❌ Error MLflow: {e}")

---
# 🖼️ Sección 3: El dataset CIFAR-10

## ¿Qué es CIFAR-10?

CIFAR-10 es un benchmark clásico de visión computacional con:
- **60,000 imágenes** en color de 32×32 píxeles
- **10 clases** balanceadas: avión, automóvil, pájaro, gato, ciervo, perro, rana, caballo, barco, camión
- **50,000 para entrenamiento** y **10,000 para prueba**

```
Imagen CIFAR-10: tensor de forma (32, 32, 3)
                  ↑     ↑    ↑
                alto  ancho  canales RGB
```

> 💡 Para la clase usaremos un **subconjunto reducido** para que el GridSearch
> tarde minutos y no horas. Puedes cambiar `N_TRAIN` y `N_TEST` para usar el dataset completo.

In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Nombres de las clases en español
CLASES = ["avión", "automóvil", "pájaro", "gato", "ciervo",
          "perro",  "rana",      "caballo", "barco", "camión"]

# ── Cargar dataset completo ───────────────────────────────────
(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()

# ── Subconjunto para la clase (cambia a None para usar todo) ──
N_TRAIN = 10_000   # de 50,000 disponibles
N_TEST  =  2_000   # de 10,000 disponibles

X_train = X_train_full[:N_TRAIN]
y_train = y_train_full[:N_TRAIN]
X_test  = X_test_full[:N_TEST]
y_test  = y_test_full[:N_TEST]

# ── Normalizar píxeles a [0, 1] ───────────────────────────────
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# ── One-hot encoding de las etiquetas ────────────────────────
y_train_ohe = to_categorical(y_train, num_classes=10)
y_test_ohe  = to_categorical(y_test,  num_classes=10)

print(f"📊 Conjunto de entrenamiento: {X_train.shape}  ({N_TRAIN:,} imágenes)")
print(f"📊 Conjunto de prueba:        {X_test.shape}   ({N_TEST:,} imágenes)")
print(f"📊 Forma de una imagen:       {X_train[0].shape}")
print(f"📊 Clases:                    {len(CLASES)} → {CLASES}")

In [ ]:
# Visualizar una muestra del dataset
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle("Muestra de CIFAR-10 — 3 ejemplos por clase", fontsize=13, y=1.02)

for clase_idx in range(10):
    # Encontrar índices de esta clase
    indices = np.where(y_train.flatten() == clase_idx)[0][:3]
    for row, idx in enumerate(indices):
        ax = axes[row, clase_idx]
        ax.imshow(X_train[idx])
        ax.axis('off')
        if row == 0:
            ax.set_title(CLASES[clase_idx], fontsize=9)

plt.tight_layout()
plt.savefig("muestra_cifar10.png", dpi=100, bbox_inches='tight')
plt.show()
print("📸 Imagen guardada como muestra_cifar10.png")

---
# 🧠 Sección 4: Arquitectura de la CNN

## ¿Qué es una CNN?

Una **Red Neuronal Convolucional** aprende filtros que detectan patrones visuales:
bordes, texturas, formas, y finalmente objetos completos.

```
Imagen (32×32×3)
      │
      ▼
 Conv2D + ReLU    ← detecta bordes y texturas simples
      │
      ▼
 MaxPooling2D     ← reduce tamaño espacial a la mitad
      │
      ▼
 Conv2D + ReLU    ← detecta patrones más complejos
      │
      ▼
 MaxPooling2D
      │
      ▼
 Flatten          ← convierte 3D → 1D
      │
      ▼
 Dense + ReLU     ← combina características
      │
      ▼
 Dropout          ← regularización (evita overfitting)
      │
      ▼
 Dense (10) + Softmax  ← probabilidad de cada clase
```

## Hiperparámetros que buscaremos con GridSearch

| Hiperparámetro | Valores a probar | ¿Qué controla? |
|---|---|---|
| `filtros` | [32, 64] | Capacidad de la red para detectar patrones |
| `learning_rate` | [0.001, 0.0001] | Tamaño de paso en el gradiente descendente |
| `dropout_rate` | [0.3, 0.5] | Fracción de neuronas desactivadas (regularización) |

Total: 2 × 2 × 2 = **8 combinaciones** — cada una es un run en MLflow.

In [ ]:
def construir_cnn(filtros=32, learning_rate=0.001, dropout_rate=0.3):
    """
    Construye y compila una CNN para clasificación en CIFAR-10.
    
    Args:
        filtros:       Número de filtros en capas convolucionales
        learning_rate: Tasa de aprendizaje del optimizador Adam
        dropout_rate:  Fracción de neuronas a desactivar (0-1)
    
    Returns:
        Modelo Keras compilado, listo para entrenar
    """
    model = keras.Sequential([
        # ── Bloque convolucional 1 ────────────────────────────
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(filtros, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(filtros, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_rate / 2),   # Menos dropout en capas conv

        # ── Bloque convolucional 2 ────────────────────────────
        layers.Conv2D(filtros * 2, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(filtros * 2, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_rate / 2),

        # ── Clasificador ──────────────────────────────────────
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),        # Más dropout en capas densas
        layers.Dense(10, activation='softmax'),
    ], name=f"CNN_f{filtros}_lr{learning_rate}_do{dropout_rate}")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Visualizar arquitectura con parámetros por defecto
modelo_demo = construir_cnn(filtros=32, learning_rate=0.001, dropout_rate=0.3)
modelo_demo.summary()
print(f"\n📊 Total parámetros entrenables: {modelo_demo.count_params():,}")

---
# ⏹️ Sección 5: EarlyStopping — ¿para qué sirve?

## El problema del overfitting en el tiempo

Sin EarlyStopping, el modelo entrenaría todas las épocas aunque ya no mejore:

```
Época  1: val_accuracy = 0.40  ← aprendiendo
Época  5: val_accuracy = 0.58  ← mejorando
Época 10: val_accuracy = 0.63  ← meseta
Época 15: val_accuracy = 0.62  ← empezando overfitting
Época 20: val_accuracy = 0.60  ← overfitting claro
                                   ← ¡aquí debió parar!
```

## Parámetros de EarlyStopping

| Parámetro | Valor | Significado |
|---|---|---|
| `monitor` | `val_accuracy` | Métrica a observar |
| `patience` | 5 | Épocas a esperar sin mejora antes de parar |
| `restore_best_weights` | `True` | Restaura los pesos de la mejor época |
| `min_delta` | 0.001 | Mejora mínima que cuenta como progreso |

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def crear_callbacks():
    """
    Crea los callbacks para el entrenamiento.
    Se llama dentro de cada run del GridSearch para tener
    instancias frescas (sin estado de entrenamientos anteriores).
    """
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=5,                  # Espera 5 épocas sin mejora
        min_delta=0.001,             # Mejora mínima considerada real
        restore_best_weights=True,   # ← Clave: vuelve a los mejores pesos
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,                  # Divide el LR a la mitad
        patience=3,                  # Después de 3 épocas sin mejorar
        min_lr=1e-6,
        verbose=1
    )

    return [early_stopping, reduce_lr]


print("✅ Callbacks configurados:")
print("   - EarlyStopping   → para cuando val_accuracy no mejora en 5 épocas")
print("   - ReduceLROnPlateau → reduce learning rate si val_loss se estanca")

---
# 🔁 Sección 6: Callback personalizado para MLflow

Keras llama a los callbacks al final de cada época.  
Creamos uno propio que registra las métricas en MLflow con `step=epoch`,
lo que genera las gráficas de curvas de entrenamiento en la UI.

In [ ]:
class MLflowCallback(keras.callbacks.Callback):
    """
    Callback que registra métricas de entrenamiento en MLflow
    al final de cada época.
    
    Esto permite ver las curvas de aprendizaje directamente
    en la interfaz web de MLflow.
    """

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        
        # Registrar cada métrica con step=epoch
        # Esto genera gráficas de curvas en MLflow UI
        mlflow.log_metric("train_loss",     logs.get('loss', 0),         step=epoch)
        mlflow.log_metric("train_accuracy", logs.get('accuracy', 0),     step=epoch)
        mlflow.log_metric("val_loss",       logs.get('val_loss', 0),     step=epoch)
        mlflow.log_metric("val_accuracy",   logs.get('val_accuracy', 0), step=epoch)

        # También registrar el learning rate actual (cambia con ReduceLROnPlateau)
        lr = float(self.model.optimizer.learning_rate)
        mlflow.log_metric("learning_rate", lr, step=epoch)


print("✅ MLflowCallback definido")
print("   Registrará por época: train_loss, train_accuracy, val_loss, val_accuracy, learning_rate")

---
# 🔎 Sección 7: GridSearch con MLflow

## ¿Por qué GridSearch manual y no `GridSearchCV` de sklearn?

`GridSearchCV` no entiende modelos de Keras con callbacks y validación por épocas.
El enfoque manual es más transparente y nos da control total sobre qué registrar.

```
param_grid = {
    filtros:       [32, 64]       → 2 opciones
    learning_rate: [0.001, 0.0001]→ 2 opciones  
    dropout_rate:  [0.3, 0.5]    → 2 opciones
}
                         ↓
              2 × 2 × 2 = 8 combinaciones
              8 runs en MLflow
```

> ⏱️ **Tiempo estimado:** ~2-5 min por run en CPU con 10K imágenes.  
> El EarlyStopping detendrá el entrenamiento antes de las 30 épocas si no mejora.

In [ ]:
import itertools
import time

# ── Espacio de búsqueda ───────────────────────────────────────
param_grid = {
    "filtros":       [32, 64],
    "learning_rate": [0.001, 0.0001],
    "dropout_rate":  [0.3, 0.5],
}

# Generar todas las combinaciones
keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
total  = len(combos)

print(f"🔎 GridSearch: {total} combinaciones a evaluar")
print(f"   Hiperparámetros: {keys}")
print()
for i, combo in enumerate(combos, 1):
    params = dict(zip(keys, combo))
    print(f"   Combinación {i:2d}: {params}")

# Configuración del entrenamiento
EPOCAS     = 30    # Máximo (EarlyStopping parará antes si no mejora)
BATCH_SIZE = 64
VAL_SPLIT  = 0.2   # 20% del train para validación

print(f"\n⚙️  Épocas máximas: {EPOCAS} (con EarlyStopping patience=5)")
print(f"⚙️  Batch size: {BATCH_SIZE}")
print(f"⚙️  Validación: {VAL_SPLIT*100:.0f}% del conjunto de entrenamiento")

In [ ]:
# ── Ejecutar el GridSearch completo ──────────────────────────
mlflow.set_experiment("cifar10-cnn-gridsearch")

resultados = []   # Guardar resultados para comparar al final
t_inicio_total = time.time()

for i, combo in enumerate(combos, 1):
    params = dict(zip(keys, combo))
    
    run_name = (
        f"f{params['filtros']}_"
        f"lr{params['learning_rate']}_"
        f"do{params['dropout_rate']}"
    )
    
    print(f"\n{'='*60}")
    print(f"▶ Run {i}/{total}: {run_name}")
    print(f"  Parámetros: {params}")
    print(f"{'='*60}")

    with mlflow.start_run(run_name=run_name) as run:

        # 1. Registrar hiperparámetros
        mlflow.log_params(params)
        mlflow.log_param("batch_size",     BATCH_SIZE)
        mlflow.log_param("epocas_max",     EPOCAS)
        mlflow.log_param("val_split",      VAL_SPLIT)
        mlflow.log_param("n_train",        N_TRAIN)
        mlflow.log_param("dataset",        "CIFAR-10")
        mlflow.log_param("early_stopping_patience", 5)

        # 2. Construir modelo con estos hiperparámetros
        modelo = construir_cnn(**params)
        mlflow.log_param("total_parametros", modelo.count_params())

        # 3. Definir callbacks (instancias frescas por cada run)
        callbacks = crear_callbacks() + [MLflowCallback()]

        # 4. Entrenar
        t0 = time.time()
        historia = modelo.fit(
            X_train, y_train_ohe,
            epochs=EPOCAS,
            batch_size=BATCH_SIZE,
            validation_split=VAL_SPLIT,
            callbacks=callbacks,
            verbose=1
        )
        tiempo_entrenamiento = time.time() - t0

        # 5. Métricas finales en el conjunto de prueba
        test_loss, test_acc = modelo.evaluate(
            X_test, y_test_ohe, verbose=0
        )

        # 6. Registrar métricas finales
        epocas_reales = len(historia.history['val_accuracy'])
        mejor_val_acc = max(historia.history['val_accuracy'])

        mlflow.log_metric("test_accuracy",         test_acc)
        mlflow.log_metric("test_loss",             test_loss)
        mlflow.log_metric("mejor_val_accuracy",    mejor_val_acc)
        mlflow.log_metric("epocas_ejecutadas",     epocas_reales)
        mlflow.log_metric("tiempo_entrenamiento_s", tiempo_entrenamiento)

        # 7. Guardar el modelo
        mlflow.keras.log_model(modelo, name="modelo_cnn")

        # 8. Guardar imagen de muestra clasificada
        from sklearn.metrics import classification_report
        y_pred = np.argmax(modelo.predict(X_test, verbose=0), axis=1)
        report = classification_report(
            y_test.flatten(), y_pred,
            target_names=CLASES, output_dict=True
        )
        # Registrar F1-score por clase
        for clase, metricas in report.items():
            if isinstance(metricas, dict) and 'f1-score' in metricas:
                nombre = clase.replace(' ', '_')
                mlflow.log_metric(f"f1_{nombre}", metricas['f1-score'])

        # Guardar resultado en lista local
        resultados.append({
            "run_name":       run_name,
            "run_id":         run.info.run_id,
            **params,
            "test_accuracy":  test_acc,
            "mejor_val_acc":  mejor_val_acc,
            "epocas":         epocas_reales,
            "tiempo_s":       tiempo_entrenamiento,
        })

        print(f"\n  ✅ Finalizado")
        print(f"     Test accuracy:    {test_acc:.4f}")
        print(f"     Mejor val acc:    {mejor_val_acc:.4f}")
        print(f"     Épocas reales:    {epocas_reales} / {EPOCAS}")
        print(f"     Tiempo:           {tiempo_entrenamiento:.0f}s")

tiempo_total = time.time() - t_inicio_total
print(f"\n{'='*60}")
print(f"🏁 GridSearch completado en {tiempo_total/60:.1f} minutos")
print(f"   {total} modelos entrenados y registrados en MLflow")

---
# 📊 Sección 8: Comparar resultados

In [ ]:
import pandas as pd

# Tabla de resultados ordenada por test_accuracy
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('test_accuracy', ascending=False)
df_resultados.index = range(1, len(df_resultados) + 1)

print("🏆 RANKING DE MODELOS — CIFAR-10 GridSearch")
print("=" * 70)
display(
    df_resultados[[
        'run_name', 'filtros', 'learning_rate', 'dropout_rate',
        'test_accuracy', 'mejor_val_acc', 'epocas', 'tiempo_s'
    ]].style.format({
        'test_accuracy': '{:.4f}',
        'mejor_val_acc': '{:.4f}',
        'tiempo_s':      '{:.0f}s'
    }).background_gradient(subset=['test_accuracy'], cmap='YlGn')
)

mejor = df_resultados.iloc[0]
print(f"\n🥇 Mejor modelo: {mejor['run_name']}")
print(f"   Test accuracy: {mejor['test_accuracy']:.4f}")
print(f"   Parámetros:    filtros={mejor['filtros']}, "
      f"lr={mejor['learning_rate']}, dropout={mejor['dropout_rate']}")

In [ ]:
# Gráfica comparativa
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Impacto de cada hiperparámetro en test_accuracy', fontsize=13)

for ax, param in zip(axes, ['filtros', 'learning_rate', 'dropout_rate']):
    grupos = df_resultados.groupby(param)['test_accuracy'].mean()
    ax.bar(
        [str(k) for k in grupos.index],
        grupos.values,
        color=['#4C72B0', '#DD8452'],
        edgecolor='gray', linewidth=0.5
    )
    ax.set_xlabel(param, fontsize=11)
    ax.set_ylabel('Test Accuracy (promedio)', fontsize=10)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(ax.patches, grupos.values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10
        )

plt.tight_layout()
plt.savefig('gridsearch_comparacion.png', dpi=100, bbox_inches='tight')
plt.show()

# Registrar la gráfica en el run del mejor modelo
with mlflow.start_run(run_id=mejor['run_id']):
    mlflow.log_artifact('gridsearch_comparacion.png')
print("📊 Gráfica guardada y registrada en MLflow")

---
# 🖥️ Sección 9: Cómo leer los resultados en MLflow UI

Ve a `https://unab-n8n.duckdns.org/mlflow/` → experimento **"cifar10-cnn-gridsearch"**

## Lo que verás:

**Vista de tabla de runs**
- 8 filas, una por combinación de hiperparámetros
- Columnas: `test_accuracy`, `mejor_val_accuracy`, `epocas_ejecutadas`
- Ordena por `test_accuracy` descendente para ver el ganador

**Vista de curvas por época (dentro de un run)**
- Pestaña "Metrics" → selecciona `train_accuracy` y `val_accuracy`
- Verás la curva de aprendizaje y el punto donde paró el EarlyStopping
- Si `val_accuracy` se separa de `train_accuracy` → overfitting

**Comparar todos los runs**
- Selecciona todos (checkbox header) → botón "Compare"
- "Parallel Coordinates" → arrastra los ejes para filtrar modelos
- Busca la línea que llega al `test_accuracy` más alto

## Preguntas de análisis

> 🤔 ¿El modelo con más filtros (64) siempre gana?  
> 🤔 ¿El EarlyStopping activó antes en los modelos con `lr=0.0001`?  
> 🤔 ¿Qué combinación tiene la mejor relación accuracy / tiempo de entrenamiento?

---
# 🏋️ Sección 10: Ejercicios propuestos

### Ejercicio 1 — Ampliar el grid
Agrega `batch_size: [32, 128]` al `param_grid`.  
¿Cuántas combinaciones totales quedan? ¿Cómo cambia el tiempo de entrenamiento?

---

### Ejercicio 2 — Data augmentation
Antes del entrenamiento, aplica aumentación de datos:
```python
datagen = keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
)
```
Registra si la mejor combinación del GridSearch mejora con augmentation.

---

### Ejercicio 3 — Dataset completo
Cambia `N_TRAIN = None` para usar las 50,000 imágenes completas.  
Ejecuta solo el mejor modelo del GridSearch. ¿Cuánto mejora el accuracy?

---

### Ejercicio 4 — Cargar el mejor modelo desde MLflow
```python
run_id = "pega-aqui-el-run-id-del-mejor-modelo"
modelo_cargado = mlflow.keras.load_model(f"runs:/{run_id}/modelo_cnn")
test_loss, test_acc = modelo_cargado.evaluate(X_test, y_test_ohe)
print(f"Accuracy cargado desde MLflow: {test_acc:.4f}")
```
¿El accuracy es idéntico al registrado en el run?

---

### Ejercicio 5 — Matriz de confusión
Con el mejor modelo, calcula y visualiza la matriz de confusión.  
¿Qué clases confunde más el modelo? ¿Tiene sentido visualmente?
```python
from sklearn.metrics import ConfusionMatrixDisplay
```

---
# 🎓 Resumen

| Concepto | Lo que hiciste |
|---|---|
| CIFAR-10 | Cargaste, normalizaste y visualizaste el dataset |
| CNN | Construiste una red con 2 bloques conv + clasificador denso |
| EarlyStopping | Evitaste overfitting deteniendo el entrenamiento a tiempo |
| ReduceLROnPlateau | Ajustaste el learning rate automáticamente |
| MLflowCallback | Registraste curvas de entrenamiento época por época |
| GridSearch manual | Evaluaste 8 combinaciones de hiperparámetros |
| MLflow runs | Cada combinación quedó como un run comparable en la UI |
| Análisis | Identificaste qué hiperparámetros tienen más impacto |

---

## 🔗 Recursos

- [CIFAR-10 paper original](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf)
- [Keras CNN guide](https://keras.io/guides/)
- [MLflow Keras integration](https://mlflow.org/docs/latest/python_api/mlflow.keras.html)
- [EarlyStopping docs](https://keras.io/api/callbacks/early_stopping/)

---

> 🚀 **Siguiente módulo:** Transfer Learning con modelos preentrenados (ResNet, EfficientNet)
> registrando los experimentos de fine-tuning en MLflow.